# ASNN-Goose v15 RunPod Operator

## Purpose
- Use this notebook on RunPod to stage and execute Phase B without forking the canonical research logic.
- The canonical logic notebook remains `notebooks/asnn_goose_v15_reset_master.ipynb`.
- This notebook only sets parameters, validates the environment, and executes the canonical notebook in a fresh subprocess.

## Workflow
1. Edit the config cell.
2. Run the preflight cell and fix any structural problem it reports.
3. Set `CONFIRM_EXECUTION = True`.
4. Run the execution cell for `SMOKE`, then `DIAGNOSE`, then `FULL`.
5. Inspect the artifact cell.
6. Bring the dossier bundle back to the laptop and register it locally.

## Guardrails
- Keep `GERHARD_ENABLE_REGISTER_RUN = False` in this operator notebook.
- Do not modify model math or notebook thresholds here.
- If `SMOKE` or `DIAGNOSE` fails structurally, stop and inspect the partial output bundle.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    import psutil
except ModuleNotFoundError:
    psutil = None


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/workspace/gerhard'), Path('/workspace')]
    seen = set()
    for candidate in candidates:
        candidate_text = str(candidate)
        if candidate_text in seen:
            continue
        seen.add(candidate_text)
        if (
            (candidate / 'notebooks' / 'asnn_goose_v15_reset_master.ipynb').exists()
            and (candidate / 'scripts' / 'register_dossier_run.py').exists()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not locate repo root containing notebooks/asnn_goose_v15_reset_master.ipynb '
        'and scripts/register_dossier_run.py.'
    )


def build_run_id(run_mode: str) -> str:
    suffix = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    prefixes = {
        'SMOKE': 'v15_preflight_smoke',
        'DIAGNOSE': 'v15_preflight_diagnose',
        'FULL': 'v15_full',
    }
    return f"{prefixes[run_mode]}_{suffix}"


REPO_ROOT = find_repo_root()
RESET_NOTEBOOK = REPO_ROOT / 'notebooks' / 'asnn_goose_v15_reset_master.ipynb'
RUN_ARTIFACTS_ROOT = REPO_ROOT / 'outputs'
OPERATOR_OUTPUT_ROOT = RUN_ARTIFACTS_ROOT / 'operator_runs'
OPERATOR_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f'Repo root: {REPO_ROOT}')
print(f'Canonical reset notebook: {RESET_NOTEBOOK}')


In [ ]:
# Edit these values before running the preflight cell.

RUN_MODE = 'SMOKE'  # SMOKE, DIAGNOSE, FULL
RUN_ID = build_run_id(RUN_MODE)
CHECKPOINT_PATH = '/absolute/path/to/your/checkpoint.pt'

GERHARD_BATCH_SIZE = 8
GERHARD_SMOKE_BATCHES = 2
GERHARD_FULL_BATCHES_SMOKE = 4
GERHARD_FULL_BATCHES_DIAGNOSE = 40
GERHARD_FULL_BATCHES = 20

GERHARD_ENABLE_DOSSIER_EXPORT = True
GERHARD_ENABLE_AUTODOWNLOAD_DOSSIER = False  # headless nbconvert execution
GERHARD_ENABLE_REGISTER_RUN = False

CONFIRM_EXECUTION = False

if RUN_MODE not in {'SMOKE', 'DIAGNOSE', 'FULL'}:
    raise ValueError(f'RUN_MODE must be one of SMOKE/DIAGNOSE/FULL, got: {RUN_MODE}')

EXECUTION_ENV = os.environ.copy()
EXECUTION_ENV.update(
    {
        'GERHARD_RUN_MODE': RUN_MODE,
        'GERHARD_RUN_ID': RUN_ID,
        'GERHARD_CHECKPOINT_PATH': CHECKPOINT_PATH,
        'GERHARD_BATCH_SIZE': str(GERHARD_BATCH_SIZE),
        'GERHARD_SMOKE_BATCHES': str(GERHARD_SMOKE_BATCHES),
        'GERHARD_FULL_BATCHES_SMOKE': str(GERHARD_FULL_BATCHES_SMOKE),
        'GERHARD_FULL_BATCHES_DIAGNOSE': str(GERHARD_FULL_BATCHES_DIAGNOSE),
        'GERHARD_FULL_BATCHES': str(GERHARD_FULL_BATCHES),
        'GERHARD_ENABLE_DOSSIER_EXPORT': '1' if GERHARD_ENABLE_DOSSIER_EXPORT else '0',
        'GERHARD_ENABLE_AUTODOWNLOAD_DOSSIER': '1' if GERHARD_ENABLE_AUTODOWNLOAD_DOSSIER else '0',
        'GERHARD_ENABLE_REGISTER_RUN': '1' if GERHARD_ENABLE_REGISTER_RUN else '0',
    }
)

RUN_DIR = RUN_ARTIFACTS_ROOT / RUN_ID
EXECUTED_NOTEBOOK_DIR = OPERATOR_OUTPUT_ROOT / RUN_ID
EXECUTED_NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)
EXECUTED_NOTEBOOK_NAME = f'{RUN_ID}_{RUN_MODE.lower()}_executed.ipynb'
ENV_SNAPSHOT_PATH = EXECUTED_NOTEBOOK_DIR / 'operator_env.json'
ENV_SNAPSHOT_PATH.write_text(
    json.dumps(
        {k: EXECUTION_ENV[k] for k in sorted(EXECUTION_ENV) if k.startswith('GERHARD_')},
        indent=2,
    ),
    encoding='utf-8',
)

print(f'RUN_MODE: {RUN_MODE}')
print(f'RUN_ID: {RUN_ID}')
print(f'Checkpoint: {CHECKPOINT_PATH}')
print(f'Run artifact dir: {RUN_DIR}')
print(f'Operator snapshot: {ENV_SNAPSHOT_PATH}')
print('Set CONFIRM_EXECUTION = True only after the preflight cell passes.')


In [ ]:
# Preflight validation and command build.

if not RESET_NOTEBOOK.exists():
    raise FileNotFoundError(f'Canonical reset notebook not found: {RESET_NOTEBOOK}')

if not CHECKPOINT_PATH or CHECKPOINT_PATH == '/absolute/path/to/your/checkpoint.pt':
    raise FileNotFoundError('Set CHECKPOINT_PATH to your real checkpoint path before continuing.')

checkpoint_file = Path(CHECKPOINT_PATH).expanduser()
if not checkpoint_file.exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_file}')

jupyter_exe = shutil.which('jupyter')
if jupyter_exe:
    NBCONVERT_COMMAND = [jupyter_exe]
else:
    NBCONVERT_COMMAND = [sys.executable, '-m', 'jupyter']

NBCONVERT_COMMAND += [
    'nbconvert',
    '--to', 'notebook',
    '--execute', str(RESET_NOTEBOOK),
    '--ExecutePreprocessor.timeout=-1',
    '--output', EXECUTED_NOTEBOOK_NAME,
    '--output-dir', str(EXECUTED_NOTEBOOK_DIR),
]

print('Preflight passed.')
print(f'Checkpoint file: {checkpoint_file}')
print(f'Executed notebook path: {EXECUTED_NOTEBOOK_DIR / EXECUTED_NOTEBOOK_NAME}')
print('Execution command:')
print(' '.join(str(part) for part in NBCONVERT_COMMAND))
print('\nMemory / hardware snapshot:')

if psutil is not None:
    vm = psutil.virtual_memory()
    print(f'  RAM total GB: {vm.total / (1024 ** 3):.2f}')
    print(f'  RAM available GB: {vm.available / (1024 ** 3):.2f}')
else:
    print('  psutil not available; skipping RAM snapshot.')

try:
    gpu_probe = subprocess.run(
        [
            'nvidia-smi',
            '--query-gpu=name,memory.total,memory.used',
            '--format=csv,noheader',
        ],
        capture_output=True,
        check=True,
        text=True,
    )
    print('  ' + gpu_probe.stdout.strip().replace('\n', '\n  '))
except Exception as exc:
    print(f'  GPU probe unavailable: {exc}')

print('\nRunPod terminal monitor: watch -n 30 nvidia-smi')


In [ ]:
# Execute the canonical reset notebook in a fresh subprocess.

if not CONFIRM_EXECUTION:
    raise RuntimeError(
        'Set CONFIRM_EXECUTION = True in the config cell after reviewing the preflight output.'
    )

print(f'Starting {RUN_MODE} for run_id={RUN_ID}')
print(f'Canonical notebook: {RESET_NOTEBOOK}')
print(f'Artifact dir: {RUN_DIR}')
print('Open a RunPod terminal and run `watch -n 30 nvidia-smi` while this cell executes.')

completed = subprocess.run(
    NBCONVERT_COMMAND,
    cwd=str(REPO_ROOT),
    env=EXECUTION_ENV,
    check=False,
)

if completed.returncode != 0:
    raise RuntimeError(
        f'Canonical reset notebook execution failed with exit code {completed.returncode}.'
    )

print(f'Execution finished. Executed notebook saved to: {EXECUTED_NOTEBOOK_DIR / EXECUTED_NOTEBOOK_NAME}')


In [ ]:
# Inspect the run artifacts produced by the canonical reset notebook.

EXPECTED_ARTIFACTS = [
    'eval_suite.json',
    'metrics.json',
    'config.yaml',
    'seed.txt',
    'v15_spikingbrain.json',
    f'run_dossier_{RUN_ID}.html',
]

missing = []
for name in EXPECTED_ARTIFACTS:
    path = RUN_DIR / name
    exists = path.exists()
    print(f'{name}: {"present" if exists else "missing"} -> {path}')
    if not exists:
        missing.append(name)

if missing:
    print('\nStructural stop: missing expected artifacts.')
    print('Stop here and inspect the partial output bundle before moving to the next mode.')
else:
    dossier_path = RUN_DIR / f'run_dossier_{RUN_ID}.html'
    print('\nArtifact bundle looks structurally complete.')
    print('Laptop-side registration command:')
    print(f'python scripts/register_dossier_run.py --dossier "{dossier_path}" --phase B')


## Next Steps
- After a clean `SMOKE`, change `RUN_MODE` to `DIAGNOSE`, let `RUN_ID` regenerate, rerun the config cell, preflight cell, execution cell, and artifact cell.
- After a structurally clean `DIAGNOSE`, repeat the same flow with `RUN_MODE = 'FULL'`.
- If GPU memory is tighter than expected, reduce only `GERHARD_BATCH_SIZE` from `8` to `4`, then to `2`.
- After `FULL`, copy the dossier bundle back to the laptop and run:
  - `python scripts/register_dossier_run.py --dossier <path_to_run_dossier_<run_id>.html> --phase B`
- Then inspect:
  - `reports/index.md`
  - `state/program_status.yaml`
  - `state/gate_results.yaml`
  - `docs/ops/STATUS_BOARD.md`
